**Applying Machine Learning Methods - Milestone 2 **

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option('display.max_columns', None)

## 1. Dataset for ML

The cleaned merged dataset from the previous phase is used for machine learning.

In [ ]:
import re

def clean_text(value):
    # This function standardizes song and artist names.
    if pd.isna(value):
        return np.nan
    value = str(value).lower().strip()
    value = re.sub(r"\(.*?\)|\[.*?\]", "", value)
    value = re.sub(r"feat\..*|ft\..*|featuring.*", "", value)
    value = re.sub(r"[^a-z0-9 ]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

def find_artist_column(df):
    # This function finds a possible artist column.
    possible_cols = ["artist_name", "artist", "artists", "track_artist", "Artist"]
    for col in possible_cols:
        if col in df.columns:
            return col
    return None

def to_number(series):
    # This function converts messy numeric columns to numbers.
    return pd.to_numeric(series.astype(str).str.replace(',', '', regex=False), errors='coerce')

# Load original files from the project folder
tiktok = pd.read_csv("tiktok.csv")
spotify = pd.read_csv("Spotify_Youtube.csv")
instagram = pd.read_csv("Trend Data Dashboard - Instagram_Snapchat.csv")

spotify = spotify.rename(columns={
    "Track": "track_name",
    "Artist": "artist_name",
    "Stream": "spotify_streams",
    "Views": "youtube_views"
})

instagram = instagram.rename(columns={
    "Music": "track_name",
    "Artist": "artist_name"
})
if "release_date" in tiktok.columns:
    tiktok["release_date"] = tiktok["release_date"].astype(str)
    tiktok["tiktok_year"] = pd.to_numeric(tiktok["release_date"].str[:4], errors="coerce")
    tiktok = tiktok[tiktok["tiktok_year"] >= 2019].copy()
for data in [tiktok, spotify, instagram]:
    data["track_clean"] = data["track_name"].apply(clean_text)
    artist_col = find_artist_column(data)
    if artist_col is not None:
        data["artist_clean"] = data[artist_col].apply(clean_text)
if "artist_clean" in tiktok.columns and "artist_clean" in spotify.columns:
    merge_keys = ["track_clean", "artist_clean"]
else:
    merge_keys = ["track_clean"]
tiktok_merge = tiktok.drop_duplicates(subset=merge_keys).copy()
spotify_merge = spotify.drop_duplicates(subset=merge_keys).copy()
df = pd.merge(
    tiktok_merge,
    spotify_merge,
    on=merge_keys,
    how="inner",
    suffixes=("_tiktok", "_spotify")
)
print("Merge keys used:", merge_keys)
print("Merged dataframe shape:", df.shape)
if "track_clean" in instagram.columns:
    if "artist_clean" in instagram.columns and "artist_clean" in df.columns:
        instagram_keys = ["track_clean", "artist_clean"]
    else:
        instagram_keys = ["track_clean"]
    instagram_merge = instagram.drop_duplicates(subset=instagram_keys).copy()
    df = pd.merge(df, instagram_merge, on=instagram_keys, how="left", suffixes=("", "_instagram"))
print("Final dataframe shape:", df.shape)
df.head()

## 2. Feature Selection

TikTok-related variables are used to predict Spotify streams.

In [ ]:
# Convert likely numeric columns
for col in df.columns:
    if col not in ["track_clean", "artist_clean"]:
        converted = to_number(df[col])
        if converted.notna().sum() > 0:
            df[col] = converted

target_col = "spotify_streams"

candidate_features = ["popularity", "youtube_views", "tiktok_year"]
extra_keywords = ["like", "view", "share", "comment", "rank", "score"]

for col in df.columns:
    lower_col = col.lower()
    if any(key in lower_col for key in extra_keywords):
        if col != target_col and pd.api.types.is_numeric_dtype(df[col]):
            candidate_features.append(col)

features = []
for col in candidate_features:
    if col in df.columns and col not in features and col != target_col:
        features.append(col)

features = [col for col in features if pd.api.types.is_numeric_dtype(df[col])]

ml_df = df[[target_col] + features].replace([np.inf, -np.inf], np.nan).dropna().copy()
ml_df["log_spotify_streams"] = np.log1p(ml_df[target_col])

print("Features used:", features)
print("ML dataframe shape:", ml_df.shape)
ml_df.head()

## 3. Regression Models



In [ ]:
#Linear Regression is used as a baseline model. Random Forest is also applied to capture non-linear patterns
X = ml_df[features]
y = ml_df["log_spotify_streams"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Linear Regression")
print("R2:", r2_score(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("
Random Forest")
print("R2:", r2_score(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("MAE:", mean_absolute_error(y_test, y_pred_rf))

Random Forest is expected to perform better if the relationship is not purely linear.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_rf, alpha=0.6)
plt.xlabel("Actual log Spotify streams")
plt.ylabel("Predicted log Spotify streams")
plt.title("Random Forest: Actual vs Predicted")
plt.show()

## 4. Feature Importance

Feature importance is used to see which variables affect the model more.

In [ ]:
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Feature Importance")
plt.show()

## 5. Classification Model

A classification model is used to predict whether a song is a hit or not.

In [ ]:
ml_df["hit"] = (ml_df[target_col] > ml_df[target_col].median()).astype(int)

X = ml_df[features]
y = ml_df["hit"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("
Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 6. Clustering

K-Means is used to find groups of songs with similar popularity patterns.

In [ ]:
X_cluster = ml_df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)
inertia = []
k_values = range(1, 8)
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
plt.figure(figsize=(7, 5))
plt.plot(k_values, inertia, marker='o')
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
ml_df["cluster"] = kmeans.fit_predict(X_scaled)
print(ml_df["cluster"].value_counts())
cluster_summary = ml_df.groupby("cluster")[[target_col] + features].mean()
cluster_summary

## 7. Conclusion

TikTok-related features can be used to predict Spotify streams to some extent. The model is limited, but the results show that social media popularity has some relationship with streaming performance.